In [ ]:
# ==========================================
# 1. SETUP GOOGLE COLAB & MOUNT DRIVE
# ==========================================
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ==========================================
# 2. IMPORT PACKAGES
# ==========================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# ==========================================
# 3. DATA LOADING
# ==========================================
train_df = pd.read_csv('/content/drive/My Drive/Colab Notebooks/DataSet/train.csv')
test_df = pd.read_csv('/content/drive/My Drive/Colab Notebooks/DataSet/test.csv')
#df = pd.concat([train_df, test_df], ignore_index=True)
df = train_df
print("Data Shape after combination:", df.shape)

In [ ]:
df.head(), df.shape

In [ ]:
# Statistical summary for numeric columns (Weight, Price, Sales, etc.)
df.describe()

In [ ]:
# Check data types (Ensure Price and Sales are floats/numbers)
print(df.info())

# **Missing Values Per Column - Bar Chart **

In [ ]:
# ==========================================
# Missing Value Analysis
# ==========================================
missing_data = df.isnull().sum()
missing_percent = (missing_data / len(df)) * 100
missing_df = pd.DataFrame({'Total Missing': missing_data, 'Percentage': missing_percent})
print("--- Missing Values Summary ---")
print(missing_df[missing_df['Total Missing'] > 0])

# ==========================================
# Inconsistent Label Detection (Fat column)
# ==========================================
fat_counts = df['Fat'].value_counts()
print("\n--- Inconsistent Labels in 'Fat' Column ---")
print(fat_counts)

# ==========================================
# Numeric Range Analysis (Red Flags)
# ==========================================
numeric_cols = ['Weight', 'Visibility', 'Price', 'OperationYear', 'Sales']
range_summary = df[numeric_cols].agg(['min', 'max', 'mean', 'median'])
print("\n--- Numeric Range Summary (Spotting Red Flags) ---")
print(range_summary)

# ==========================================
# VISUALIZATIONS
# ==========================================
# 1. Missing Data Bar Chart
plt.figure(figsize=(10, 6))
missing_data.plot(kind='bar', color='salmon')
plt.title('Missing Values per Column (Pre-Cleaning)')
plt.ylabel('Count')
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('missing_values_pre.png')

# **Missing Values Count Weight & Size**

In [ ]:
plt.figure(figsize=(10, 6))
missing_data = df.isnull().sum()
missing_data = missing_data[missing_data > 0] # Only show columns with gaps
sns.barplot(x=missing_data.index, y=missing_data.values, palette='Reds_r')
plt.title('Missing Values Count (Gaps in Data)')
plt.ylabel('Number of Missing Rows')
plt.xlabel('Columns / Features')
plt.savefig('diagnostic_missing_values.png')

# **Categorical Label Frequency**

In [ ]:
plt.figure(figsize=(8, 5))
sns.countplot(data=df, x='Fat', palette='Set2', order=df['Fat'].value_counts().index)
plt.title('Label Inconsistency in "Fat" Column')
plt.xlabel('Fat Labels')
plt.savefig('diagnostic_label_inconsistency.png')

# **Numeric Box Plots**

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 12))
# ==========================================
# Weight: Look for negative values
# ==========================================
sns.boxplot(ax=axes[0, 0], x=df['Weight'], color='skyblue').set_title('Weight (Note Negatives)')

# ==========================================
# Price: Look for the $35M outlier
# ==========================================
sns.boxplot(ax=axes[0, 1], x=df['Price'], color='lightgreen').set_title('Price (Note extreme outliers)')

# ==========================================
# Sales: Look for the $100M extreme
# ==========================================
sns.boxplot(ax=axes[1, 0], x=df['Sales'], color='salmon').set_title('Sales (Extreme skewness)')

# ==========================================
# OperationYear: Look for "Future" years
# ==========================================
sns.boxplot(ax=axes[1, 1], x=df['OperationYear'], color='gold').set_title('OperationYear (Note year 3678)')

plt.tight_layout()
plt.savefig('diagnostic_numeric_red_flags.png')

# **Distribution Histogram**

In [ ]:
plt.figure(figsize=(10, 6))
sns.histplot(df['Sales'], bins=50, kde=True, color='purple')
plt.title('Raw Sales Distribution')
plt.savefig('diagnostic_sales_histogram.png')

# **Raw Correlation Heatmap**

In [ ]:
plt.figure(figsize=(10, 8))
numeric_df = df.select_dtypes(include=['number'])
sns.heatmap(numeric_df.corr(), annot=True, cmap='coolwarm', fmt=".2f")
plt.title('Correlation Heatmap')
plt.savefig('diagnostic_correlation_raw.png')

# **EDA Approaching**

**Clean up placeholder symbols like '-'**

In [ ]:
# ==========================================
# Standardize 'Fat' column labels
# ==========================================
fat_map = {'low fat': 'Low Fat', 'LF': 'Low Fat', 'reg': 'Regular', 'Low Fat': 'Low Fat', 'Regular': 'Regular'}
df['Fat'] = df['Fat'].map(fat_map)

In [ ]:
# ==========================================
# Handle Missing Values
# ==========================================
df['Size'] = df['Size'].fillna('Unknown')

In [ ]:
# ==========================================
# Convert 'Impossible' values to NaN (Not a Number)
# ==========================================
df.loc[(df['OperationYear'] < 1900) | (df['OperationYear'] > 2025), 'OperationYear'] = np.nan

In [ ]:
# ==========================================
# Use MODE for categorical/discrete (Size, OperationYear)
# ==========================================
df['OperationYear'] = df['OperationYear'].fillna(df['OperationYear'].mode()[0])

In [ ]:
# ==========================================
# Verify: All 8,540 rows are preserved
# ==========================================
print(f"Total rows remaining: {len(df)}")

In [ ]:
# ==========================================
# Quick EDA: Identifying the 'Impossible' values
# ==========================================
print("--- Raw Statistics for Price and Sales ---")
print(df[['Price', 'Sales']].describe())

In [ ]:
# ==========================================
# Filter out Noise (Outliers & Impossible values)
# ==========================================
df_clean = df[
    (df['Weight'] > 0) & (df['Weight'] < 100) &
    (df['Price'] > 0) & (df['Price'] < 500) &
    (df['Sales'] > 0) & (df['Sales'] < 20000) &
    (df['OperationYear'] >= 1980) & (df['OperationYear'] <= 2025) &
    (df['Visibility'] >= 0) & (df['Visibility'] <= 0.4)
].copy()

In [ ]:
df_clean.info(), df_clean.shape

In [ ]:
# ==========================================
# Fill missing Weights with the median
# ==========================================
df_clean['Weight'] = df_clean['Weight'].fillna(df_clean['Weight'].median())

In [ ]:
df_clean.info(), df_clean.shape

# **# --- VISUALIZATIONS ---**

***Univaritate - Sales Distribbution***

---



In [ ]:
# ==========================================
# Visualizing the Target Variable: Sales
# ==========================================
plt.figure(figsize=(10, 5))
sns.histplot(df_clean['Sales'], kde=True, color='teal')
plt.title('Distribution of Sales')
plt.show()
plt.savefig('sales_dist.png')

***Univariate - Counting the Categories***

---



In [ ]:
# ==========================================
# Counting the Categories
# ==========================================
plt.figure(figsize=(12, 6))
sns.countplot(data=df_clean, y='Category', order=df_clean['Category'].value_counts().index, palette='viridis')
# plt.xticks(rotation=45)
plt.title('Count of Products by Category')
#plt.show()
plt.tight_layout()
plt.savefig('category_counts.png')

**Univariate - Price Range Analysis**

In [ ]:
# ==========================================
# Price Range Analysis
# ==========================================
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(10, 4))
sns.boxplot(data=df, x='Price', color='skyblue')
plt.title('Price Tier Analysis: Distribution of Item Costs')
plt.xlabel('Item Price ($)')
plt.show()

**Univariate - Sales Analysis**

In [ ]:
# ==========================================
# Sales (Histogram + KDE)
# ==========================================
sns.set_theme(style="whitegrid")

plt.figure(figsize=(10, 6))
sns.histplot(data=df, x='Sales', kde=True, color='blue', bins=30)
plt.title('Target Variable Distribution: Retail Sales')
plt.xlabel('Total Sales ($)')
plt.ylabel('Frequency (Number of Transactions)')
plt.show()

**Univariate - Visibility Graph**

In [ ]:
# ==========================================
# Visibility Graph
# ==========================================
sns.kdeplot(data=df, x='Visibility', fill=True, color='orange')
plt.title('Item Visibility Distribution')
plt.xlabel('Visibility Percentage')
plt.ylabel('Density')
plt.show()

**Univariate - Count of Store Types**

In [ ]:
# ==========================================
# Count of Store Types
# ==========================================
plt.figure(figsize=(10, 6))
sns.countplot(data=df, x='Type', order=df['Type'].value_counts().index, palette='crest')
plt.title('Count of Store Types')
plt.xlabel('Store Format')
plt.ylabel('Total Count')
plt.xticks(rotation=45)
plt.show()

***BIVARIATE: Price vs Sales (scatterplot)***

---



In [ ]:
# ==========================================
# BIVARIATE: Price vs Sales
# ==========================================
plt.figure(figsize=(10, 6))
sns.scatterplot(data=df_clean, x='Price', y='Sales', alpha=0.3)
plt.title('Price vs Sales')
plt.savefig('price_vs_sales.png')

**BIVARIATE: Type vs. Sales (Bar Chart)**

In [ ]:
# ==========================================
# BIVARIATE: Type vs. Sales
# ==========================================
plt.figure(figsize=(10, 5))
sns.barplot(data=df_clean, x='Type', y='Sales', estimator='sum', palette='viridis')
plt.title('Revenue Generation by Store Type')
plt.savefig('revenue_by_store_type.png')
plt.show()

**BIVARIATE: OperationYear vs. Sales (Line Chart)**

In [ ]:
# ==========================================
# BIVARIATE: OperationYear vs. Sales
# ==========================================
plt.figure(figsize=(10, 5))
sns.lineplot(data=df_clean, x='OperationYear', y='Sales', estimator='sum', marker='o', color='teal')
plt.title('Total Sales Growth by Store Opening Year')
plt.savefig('sales_by_operation_year.png')
plt.show()

***BIVARIATE: Category vs Sales (boxplot)***

---



In [ ]:
# ==========================================
# BIVARIATE: Category vs Sales (boxplot)
# ==========================================
plt.figure(figsize=(12, 6))
sns.boxplot(data=df_clean, x='Category', y='Sales', palette='viridis')
plt.xticks(rotation=45)
plt.title('Sales Distribution by Category')
plt.tight_layout()
plt.savefig('category_vs_sales.png')

In [ ]:
# ==========================================
# BIVARIATE: Total Sales by Category
# ==========================================
plt.figure(figsize=(12, 6))
sns.barplot(data=df_clean, x='Category', y='Sales', estimator=sum, palette='viridis')
plt.xticks(rotation=45)
plt.title('Total Sales by Category')
plt.show()

***BIVARIATE: Visibility vs Sales***

---



In [ ]:
# ==========================================
# BIVARIATE: Product Visibility vs. Sales
# ==========================================
plt.figure(figsize=(10, 6))
sns.scatterplot(data=df_clean, x='Visibility', y='Sales', alpha=0.5)
plt.title('Product Visibility vs. Sales')
plt.show()

***BIVARIATE: OperationYear vs Sales***

---



In [ ]:
# ==========================================
# BIVARIATE: OperationYear vs. Sales (Line Chart)
# ==========================================
plt.figure(figsize=(10, 5))
sns.lineplot(data=df_clean, x='OperationYear', y='Sales', estimator='sum', marker='o', color='teal')
plt.title('Total Sales Growth by Store Opening Year')
plt.show()

***BIVARIATE: Types vs Sales***

---



In [ ]:
# ==========================================
# BIVARIATE: Type vs. Sales (Bar Chart)
# ==========================================
plt.figure(figsize=(10, 5))
sns.barplot(data=df_clean, x='Type', y='Sales', estimator='sum', palette='viridis')
plt.title('Revenue Generation by Store Type')
plt.show()

***BIVARIATE: Category vs Price***

---



In [ ]:
# ==========================================
# BIVARIATE: Category vs. Price (Box Plot)
# ==========================================
plt.figure(figsize=(12, 6))
sns.boxplot(data=df_clean, x='Category', y='Price', palette='pastel')
plt.xticks(rotation=45)
plt.title('Price Distribution per Category')
plt.show()

***BIVARIATE: Security vs Sales***

---



In [ ]:
# ==========================================
# BIVARIATE: Security vs. Sales (Bar Chart)
# ==========================================
plt.figure(figsize=(10, 5))
sns.barplot(data=df_clean, x='Security', y='Sales', estimator='sum', palette='magma')
plt.title('Total Sales by Security Provider')
plt.show()

***BIVARIATE: StoreCode vs Sales***

---



In [ ]:
# ==========================================
# BIVARIATE: Total Sales by Store Code
# ==========================================
plt.figure(figsize=(12, 5))
order = df_clean.groupby('StoreCode')['Sales'].sum().sort_values(ascending=False).index
sns.barplot(data=df_clean, x='StoreCode', y='Sales', estimator='sum', order=order, palette='coolwarm')
plt.title('Ranking: Total Sales by Store Code')
plt.show()

***BIVARIATE: Weight vs. Visibility***

---



In [ ]:
# ==========================================
# BIVARIATE: Weight vs. Visibility (Scatter Plot)
# ==========================================
plt.figure(figsize=(10, 6))
sns.scatterplot(data=df_clean, x='Weight', y='Visibility', alpha=0.3)
plt.title('Weight vs. Visibility')
plt.show()

***BIVARIATE: Area vs. Sales***

---



In [ ]:
# ==========================================
# BIVARIATE: otal Sales by Area Tier
# ==========================================
plt.figure(figsize=(10, 5))
sns.barplot(data=df_clean, x='Area', y='Sales', estimator=sum, palette='viridis')
plt.title('Total Sales by Area Tier')
plt.show()

***BIVARIATE: Size vs. Sales***

---



In [ ]:
# ==========================================
# BIVARIATE: Sales Distribution by Store Size
# ==========================================
plt.figure(figsize=(10, 5))
sns.boxplot(data=df_clean, x='Size', y='Sales', palette='magma')
plt.title('Sales Distribution by Store Size')
plt.show()

***BIVARIATE: Security vs. Weight***

---



In [ ]:
# ==========================================
# BIVARIATE: Product Weight Distribution per Security Provider
# ==========================================
plt.figure(figsize=(10, 5))
sns.boxplot(data=df_clean, x='Security', y='Weight', palette='pastel')
plt.title('Product Weight Distribution per Security Provider')
plt.show()

***BIVARIATE: Type vs. Price***

---



In [ ]:
# ==========================================
# BIVARIATE: Price Range by Store Type
# ==========================================
plt.figure(figsize=(10, 5))
sns.boxplot(data=df_clean, x='Type', y='Price', palette='coolwarm')
plt.title('Price Range by Store Type')
plt.show()

***BIVARIATE: Category vs. Fat***

---



In [ ]:
# ==========================================
# BIVARIATE: Category vs. Fat (Grouped Count Plot)
# ==========================================
plt.figure(figsize=(12, 6))
sns.countplot(data=df_clean, x='Category', hue='Fat')
plt.xticks(rotation=45)
plt.title('Category vs. Fat Content')
plt.show()

***BIVARIATE: Size vs. Type***

---



In [ ]:
# ==========================================
# BIVARIATE: Size vs. Type (Heatmap)
# ==========================================
plt.figure(figsize=(8, 6))
ct = pd.crosstab(df_clean['Size'], df_clean['Type'])
sns.heatmap(ct, annot=True, fmt='d', cmap='YlGnBu')
plt.title('Size vs. Type')
plt.show()

***BIVARIATE: Security vs. Category***

---



In [ ]:
# ==========================================
# BIVARIATE: Security vs. Category (Heatmap)
# ==========================================
plt.figure(figsize=(12, 8))
ct_sec = pd.crosstab(df_clean['Security'], df_clean['Category'])
sns.heatmap(ct_sec, annot=True, fmt='d', cmap='Blues')
plt.title('Security Provider by Category')
plt.show()

**Product Visibility vs. Price (Scatterplot)**

In [ ]:
plt.figure(figsize=(10, 6))
sns.scatterplot(data=df_clean, x='Visibility', y='Price', alpha=0.5, color='coral')
plt.title('Product Visibility vs. Price')
plt.savefig('visibility_vs_price.png')
plt.show()

**Average Product Price by Store Area Tier**

In [ ]:
plt.figure(figsize=(8, 5))
sns.boxplot(data=df_clean, x='Fat', y='Price', palette='Set2')
plt.title('Price Distribution by Fat Content')
plt.savefig('price_by_fat_content.png')
plt.show()

***MULTIVARIATE: Weight & Fat vs. Sales***

---



In [ ]:
# ==========================================
# --- Weight & Fat vs. Sales (Multivariate - Scatter Plot) ---
#
# ==========================================
plt.figure(figsize=(10, 6))
sns.scatterplot(data=df_clean, x='Weight', y='Sales', hue='Fat', alpha=0.5)
plt.title('Multivariate: Sales vs Weight Segmented by Fat Content')
plt.show()

***MULTIVARIATE: Price vs Sales by Store Size***

---



In [ ]:
# ==========================================
# MULTIVARIATE: Price vs Sales by Store Size
# ==========================================
plt.figure(figsize=(12, 7))
sns.scatterplot(data=df_clean, x='Price', y='Sales', hue='Size', alpha=0.5)
plt.title('Multivariate Analysis: Price vs Sales Segmented by Store Size')
plt.savefig('multivariate_price_size_sales.png')

***MULTIVARIATE: Area & Size vs. Sales***

---



In [ ]:
# ==========================================
# Area & Size vs. Sales (Multivariate - Grouped Bar Chart) ---
#
# ==========================================
plt.figure(figsize=(12, 6))
sns.barplot(data=df_clean, x='Area', y='Sales', hue='Size', estimator='sum')
plt.title('Total Sales by Area Tier and Store Size')
plt.legend(title='Store Size', bbox_to_anchor=(1, 1))
plt.show()

***MULTIVARIATE: Category + Price + Sales***

---



In [ ]:
# ==========================================
# Category + Price + Sales (Bubble Chart)
#
# ==========================================
plt.figure(figsize=(12, 8))
stats = df_clean.groupby('Category').agg({'Price':'mean', 'Sales':'mean', 'ID':'count'}).reset_index()
sns.scatterplot(data=stats, x='Price', y='Sales', size='ID', hue='Category', sizes=(100, 2000), alpha=0.6)
plt.title('Multivariate: Price, Sales, and Volume by Category')
plt.legend(bbox_to_anchor=(1, 1))
plt.show()

***MULTIVARIATE: Size vs Type vs Sales***

---



In [ ]:
# ==========================================
# Size + Type + Sales (Grouped Bar)
# ==========================================
plt.figure(figsize=(10, 6))
sns.barplot(data=df_clean, x='Type', y='Sales', hue='Size', estimator=sum)
plt.title('Multivariate: Total Sales by Type and Size')
plt.show()

***MULTIVARIATE: Visibility + Fat + Sales***

---



In [ ]:
# ==========================================
# Visibility + Fat + Sales (Scatter with Hue)
# ==========================================
plt.figure(figsize=(10, 6))
sns.scatterplot(data=df_clean, x='Visibility', y='Sales', hue='Fat', alpha=0.4)
plt.title('Multivariate: Impact of Visibility on Sales by Fat Content')
plt.show()

***MULTIVARIATE: OperationYear + Area + Sales***

---



In [ ]:
# ==========================================
# OperationYear + Area + Sales (Line Plot with Hue)
# ==========================================
plt.figure(figsize=(10, 6))
sns.lineplot(data=df_clean, x='OperationYear', y='Sales', hue='Area', estimator=sum, marker='o')
plt.title('Multivariate: Sales Trends Over Time per Area')
plt.show()

***MULTIVARIATE: Correlation Heatmap***

---



In [ ]:
# ==========================================
# MULTIVARIATE: Correlation Heatmap
# ==========================================
plt.figure(figsize=(10, 8))
#corr = df_clean.select_dtypes(include=[np.number]).corr()
#sns.heatmap(corr, annot=True, cmap='coolwarm', fmt=".2f")
sns.heatmap(df_clean.corr(numeric_only=True), annot=True, cmap='coolwarm', fmt=".2f")
plt.title('Correlation Matrix of Numeric Features')
plt.savefig('correlation_heatmap.png')

# **Regression**

---



**Linear Regression (The Standard Baseline)**

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import LabelEncoder

In [ ]:
# ==========================================
# 2. Prepare Data
# ==========================================
le = LabelEncoder()
for col in ['Fat', 'Category', 'Security', 'StoreCode', 'Size', 'Area', 'Type']:
    df_clean[col] = le.fit_transform(df_clean[col].astype(str))

In [ ]:
X = df_clean.drop(['No', 'ID', 'Sales'], axis=1)
y = df_clean['Sales']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import LabelEncoder

# Make a copy to avoid SettingWithCopyWarning
X_train_encoded = X_train.copy()
X_test_encoded = X_test.copy()

# Identify categorical columns
categorical_cols = X_train_encoded.select_dtypes(include='object').columns

# Apply Label Encoding to categorical columns
label_encoders = {}
for col in categorical_cols:
    le = LabelEncoder()
    X_train_encoded[col] = le.fit_transform(X_train_encoded[col])
    # Use the same encoder for the test set
    X_test_encoded[col] = le.transform(X_test_encoded[col])
    label_encoders[col] = le
# ==========================================
# 3. Model Training
# ==========================================
linear_model = LinearRegression()
linear_model.fit(X_train_encoded, y_train)

# ==========================================
# 4. Evaluation
# ==========================================
preds = linear_model.predict(X_test)
print(f"--- Linear Regression Results ---")
print(f"MAE: {mean_absolute_error(y_test, preds):.2f}")
print(f"RMSE: {np.sqrt(mean_squared_error(y_test, preds)):.2f}")
print(f"R-Squared: {r2_score(y_test, preds):.4f}")

***Ridge Regression (Stable Linear Model)***

In [ ]:
from sklearn.linear_model import Ridge

In [ ]:
# ==========================================
# 3. Model Training
# ==========================================
ridge_model = Ridge(alpha=1.0) # alpha is the 'smoothness' setting
ridge_model.fit(X_train, y_train)

# 4. Evaluation
preds = ridge_model.predict(X_test)
print(f"--- Ridge Regression Results ---")
print(f"MAE: {mean_absolute_error(y_test, preds):.2f}")
print(f"RMSE: {np.sqrt(mean_squared_error(y_test, preds)):.2f}")
print(f"R-Squared: {r2_score(y_test, preds):.4f}")

***Decision Tree (The Flowchart Model)***

In [ ]:
from sklearn.tree import DecisionTreeRegressor

In [ ]:
# ==========================================
# 3. Model Training
# ==========================================
tree_model = DecisionTreeRegressor(max_depth=3, random_state=42)
tree_model.fit(X_train, y_train)


# 4. Evaluation
preds = tree_model.predict(X_test)
print(f"--- Decision Tree Results ---")
print(f"MAE: {mean_absolute_error(y_test, preds):.2f}")
print(f"RMSE: {np.sqrt(mean_squared_error(y_test, preds)):.2f}")
print(f"R-Squared: {r2_score(y_test, preds):.4f}")

In [ ]:
# ==========================================
# 1. LINEAR REGRESSION EVALUATION
# ==========================================
lr_predictions = linear_model.predict(X_test)

lr_mae = mean_absolute_error(y_test, lr_predictions)
lr_mse = mean_squared_error(y_test, lr_predictions)
lr_rmse = np.sqrt(lr_mse) # RMSE is just the square root of MSE
lr_r2 = r2_score(y_test, lr_predictions)

print("--- Linear Regression Scores ---")
print(f"MAE: {lr_mae}")
print(f"RMSE: {lr_rmse}")
print(f"R-squared: {lr_r2}")
print("\n") # This just adds a blank line to make the output look nice

In [ ]:
# ==========================================
# 2. RIDGE REGRESSION EVALUATION
# ==========================================
ridge_predictions = ridge_model.predict(X_test)

ridge_mae = mean_absolute_error(y_test, ridge_predictions)
ridge_mse = mean_squared_error(y_test, ridge_predictions)
ridge_rmse = np.sqrt(ridge_mse)
ridge_r2 = r2_score(y_test, ridge_predictions)

print("--- Ridge Regression Scores ---")
print(f"MAE: {ridge_mae}")
print(f"RMSE: {ridge_rmse}")
print(f"R-squared: {ridge_r2}")
print("\n")

In [ ]:
# ==========================================
# 3. DECISION TREE EVALUATION
# ==========================================
tree_predictions = tree_model.predict(X_test)

tree_mae = mean_absolute_error(y_test, tree_predictions)
tree_mse = mean_squared_error(y_test, tree_predictions)
tree_rmse = np.sqrt(tree_mse)
tree_r2 = r2_score(y_test, tree_predictions)

print("--- Decision Tree Scores ---")
print(f"MAE: {tree_mae}")
print(f"RMSE: {tree_rmse}")
print(f"R-squared: {tree_r2}")
print("\n")

In [ ]:
# ==========================================
# 1. LINEAR REGRESSION MSE
# ==========================================
lr_predictions = linear_model.predict(X_test)

lr_mse = mean_squared_error(y_test, lr_predictions)

In [ ]:
# ==========================================
# 1. LINEAR REGRESSION MSE
# ==========================================
lr_predictions = linear_model.predict(X_test)
lr_mse = mean_squared_error(y_test, lr_predictions)

# ==========================================
# 2. RIDGE REGRESSION MSE
# ==========================================
# Ask the Ridge model to predict sales
ridge_predictions = ridge_model.predict(X_test)

# Calculate the MSE
ridge_mse = mean_squared_error(y_test, ridge_predictions)


# ==========================================
# 3. DECISION TREE MSE
# ==========================================
# Ask the Decision Tree model to predict sales
tree_predictions = tree_model.predict(X_test)

# Calculate the MSE
tree_mse = mean_squared_error(y_test, tree_predictions)


# ==========================================
# PRINT ALL RESULTS
# ==========================================
print("--- Mean Squared Error (MSE) Scores ---")
print(f"Linear Regression MSE: {lr_mse:.2f}")
print(f"Ridge Regression MSE:  {ridge_mse:.2f}")
print(f"Decision Tree MSE:     {tree_mse:.2f}")

In [ ]:
# Import the visualization tool
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 6))
plt.scatter(y_test, tree_predictions, color='blue', alpha=0.5)

plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], color='red', linewidth=2)

plt.title('Actual Sales vs. Predicted Sales (Decision Tree Model)')
plt.xlabel('Actual Sales (Real Data)')
plt.ylabel('Predicted Sales (Model Guesses)')

plt.show()

In [ ]:
# ==========================================
# 1. CREATE THE NEW ENTRY
# ==========================================
new_product_data = {
    'Weight': [12.5],
    'Fat': [0],
    'Visibility': [0.05],
    'Category': [6],
    'Price': [150.00],
    'Security': [1],
    'StoreCode': [3],
    'OperationYear': [2015.0],
    'Size': [1],
    'Area': [2],
    'Type': [1]
}

new_entry_df = pd.DataFrame(new_product_data)


# ==========================================
# 2. PREDICT WITH ALL 3 MODELS
# ==========================================
# Ask each model to predict the sales for this new entry
lr_new_prediction = linear_model.predict(new_entry_df)
ridge_new_prediction = ridge_model.predict(new_entry_df)
tree_new_prediction = tree_model.predict(new_entry_df)


# ==========================================
# 3. PRINT THE RESULTS
# ==========================================
print("--- Predicted Sales for the New Product ---")
print(f"Linear Regression predicts: ${lr_new_prediction[0]:.2f}")
print(f"Ridge Regression predicts:  ${ridge_new_prediction[0]:.2f}")
print(f"Decision Tree predicts:     ${tree_new_prediction[0]:.2f}")